# 🛠️ Active Learning Workshop: Implementing an Inverted Matrix (Jupyter + GitHub Edition)
## 🔍 Workshop Theme
*Readable, correct, and collaboratively reviewed code—just like in the real world.*


Welcome to the 90-minute workshop! In this hands-on session, your team will build an **Inverted Index** pipeline, the foundation of many intelligent systems that need fast and relevant access to text data — such as AI agents.

### 👥 Team Guidelines
- Work in teams of 3.
- Submit one completed Jupyter Notebook per team.
- The final notebook must contain **Markdown explanations** and **Python code**.
- Push your notebook to GitHub and share the `.git` link before class ends.

---
## 🔧 Workshop Tasks Overview

1. **Document Collection**
2. **Tokenizer Implementation**
3. **Normalization Pipeline (Stemming, Stop Words, etc.)**
4. **Build and Query the Inverted Index**

> Each step includes a sample **talking point**. Your team must add your own custom **Markdown + code cells** with a **second talking point**, and test your Inverted Index with **2 phrase queries**.




## 🧠 Learning Objectives
- Implement an **Inverted Matrix** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (120 Minutes)
1. **Instructor Use Case Introduction** *(15 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(45 min)* – Complete all challenges in the notebook. Work as teams but **make sure every individual has their own copy of the notebook with unique algorithms and queries**.
3. **Push to GitHub** *(15 min)* – Individuals commit and push finished notebooks. **Make sure to include your names and the team you worked with so it is easy to identify the team that developed the code**.
4. **Instructor Review** *(30 min)* - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**, assisting with individual work.
5. **Email Delivery** *(15 min)* – Each team sends the instructor an email **with the *.git link to the GitHub repo of every team member (one email/team)**. Subject on the email is: PROG8245 - Inverted Matrix  Workshop, Team #_____.
6. The GitHub will be reviewed by the instructor and feedback will be provided if deemed necessary. Work will be assessed during the workshop.


## 💻 Submission Checklist
- ✅ `IR_InvertedMatrix_Workshop.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, and challenges coded and solved.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** and 2 phrase query tests
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `IR-invertedmatrix-workshop`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 📖 Introduction

This notebook implements a complete **Information Retrieval (IR) pipeline** built around an **Inverted Index** — the core data structure behind search engines and AI retrieval systems.

### Pipeline at a glance
1. **Document Collection** — fetch 20+ PSF blog posts (2,000+ unique words)
2. **Tokenization** — split raw text into lowercase word tokens
3. **Normalization** — remove stop words + Porter stemming
4. **Inverted Index** — `{term: [doc_id, ...]}` for Boolean retrieval
5. **Phrase Queries** — exact consecutive-token matching
6. **Positional Index** (Challenge 1) — `{term: {doc_id: [pos, ...]}}` for proximity search
7. **Memory-Efficient Positional Index** (Challenge 2) — same structure using `array` module to cut RAM by ~4×
8. **IDF** (Challenge 3) — Inverse Document Frequency weighting
9. **TF** (Challenge 4) — Term Frequency and TF-IDF scoring

### Dataset
20 blog posts from the **Python Software Foundation** RSS feed, covering releases, governance, security, and community events — stored locally in `sample_docs/`.

### Team / Individual
*(Add your name and team number here before submitting)*

## 📄 Step 1: Document Collection


### 🗣 Instructor Talking Point:
> We begin by gathering a text corpus. To build a robust index, your vocabulary should include **over 2000 unique words**. You can use scraped articles, academic papers, or open datasets.

### 🔧 Your Task:
- Collect at least 20+ text documents.
- Ensure the vocabulary exceeds 2000 unique words.
- Load the documents into a list for processing.


In [1]:
import os, requests, feedparser

def fetch_and_save_blog_posts(rss_url: str, save_folder: str, max_posts: int = 20) -> int:
    """
    Download blog posts from an RSS feed and save them as UTF-8 .txt files.

    Parameters
    ----------
    rss_url     : URL of the RSS feed.
    save_folder : Local directory to write files into (created if absent).
    max_posts   : Maximum posts to download.

    Returns
    -------
    int : Number of posts saved.

    Raises
    ------
    ValueError : If rss_url is empty or max_posts < 1.
    """
    if not rss_url or not rss_url.strip():
        raise ValueError('rss_url must be a non-empty string.')
    if max_posts < 1:
        raise ValueError(f'max_posts must be >= 1, got {max_posts}.')

    os.makedirs(save_folder, exist_ok=True)
    feed = feedparser.parse(rss_url)

    if feed.bozo and not feed.entries:
        print(f'⚠️  Feed warning: {feed.bozo_exception}. No entries found.')
        return 0

    count = 0
    for entry in feed.entries:
        if count >= max_posts:
            break
        title   = entry.get('title',   'untitled').strip()
        content = entry.get('summary', '').strip()
        if not content:
            continue
        filepath = os.path.join(save_folder, f'blog_{count + 1}.txt')
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f'{title}\n\n{content}')
        count += 1

    print(f'✅ Saved {count} blog posts to "{save_folder}"')
    return count


RSS_URL = 'https://pyfound.blogspot.com/feeds/posts/default?alt=rss'
fetch_and_save_blog_posts(RSS_URL, 'sample_docs', max_posts=20)


✅ Saved 20 blog posts to "sample_docs"


20

In [2]:
import os
import pandas as pd

def load_documents(folder_path: str) -> list:
    """
    Load all .txt files from folder_path into a list of strings.

    Files are sorted lexicographically for deterministic ordering.

    Parameters
    ----------
    folder_path : str — Path to folder containing .txt files.

    Returns
    -------
    list[str] — One string per document.

    Raises
    ------
    FileNotFoundError : If folder_path does not exist.
    ValueError        : If no .txt files are found.
    """
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f'Folder not found: "{folder_path}"')

    txt_files = sorted(f for f in os.listdir(folder_path) if f.endswith('.txt'))

    if not txt_files:
        raise ValueError(f'No .txt files in "{folder_path}". '
                         'Run fetch_and_save_blog_posts() first.')

    documents = []
    for filename in txt_files:
        with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as fh:
            text = fh.read().strip()
        if text:                         # skip completely empty files
            documents.append(text)
        else:
            print(f'  ⚠️  Skipped empty file: {filename}')

    return documents


documents = load_documents('sample_docs/')

# Pandas corpus summary
summary_df = pd.DataFrame({
    'File':   sorted(f for f in os.listdir('sample_docs/') if f.endswith('.txt'))[:len(documents)],
    'Words':  [len(d.split()) for d in documents],
    'Chars':  [len(d) for d in documents],
    'Preview':[d[:55].replace('\n',' ') + '...' for d in documents],
})
print(f'✅ Loaded {len(documents)} documents\n')
print(summary_df.to_string(index=False))
print(f'\nTotal words : {summary_df["Words"].sum():,}  |  '
      f'Avg/doc : {summary_df["Words"].mean():.0f}')


✅ Loaded 20 documents

       File  Words  Chars                                                    Preview
 blog_1.txt    620   5196 Applications to Join the PSF Meetup Pro Network Are Bac...
blog_10.txt    208   1587 PSF Code of Conduct Working Group Shares First Transpar...
blog_11.txt    750   5703 Python is for Everyone: Grab PyCharm Pro for 30% off—pl...
blog_12.txt   1310  10876 Python is for everyone: Join in the PSF year-end fundra...
blog_13.txt   1402  11443 Connecting the Dots: Understanding the PSF’s Current Fi...
blog_14.txt    452   4515 Improving security and integrity of Python package arch...
blog_15.txt   1245  10891 Open Infrastructure is Not Free: PyPI, the Python Softw...
blog_16.txt    574   4009 A new PSF Board- Another year of PSF Board Office Hour ...
blog_17.txt    877   6423 The PSF has withdrawn a $1.5 million proposal to US gov...
blog_18.txt   1250  14547 Announcing Python Software Foundation Fellow Members fo...
blog_19.txt   1047  14366 CPython Core Dev

## ✂️ Step 2: Tokenizer


### 🗣 Instructor Talking Point:
> The tokenizer breaks raw text into a stream of words (tokens). This is the foundation for every later step in IR and NLP.

### 🔧 Your Task:
- Implement a basic tokenizer that splits text into lowercase words.
- Handle punctuation removal and basic non-alphanumeric filtering.


In [3]:
import re

def tokenize(text: str) -> list:
    """
    Convert raw text into a list of lowercase word tokens.

    Uses regex \\b\\w+\\b to match word characters (letters, digits,
    underscore), discarding punctuation and whitespace. All tokens are
    lowercased for case-insensitive retrieval.

    Parameters
    ----------
    text : str — Raw document or query string.

    Returns
    -------
    list[str] — Ordered list of tokens.

    Edge cases
    ----------
    * Empty string  → returns [].
    * Non-string    → raises TypeError.
    """
    if not isinstance(text, str):
        raise TypeError(f'tokenize() expects str, got {type(text).__name__}')
    if not text.strip():
        return []   # empty / whitespace-only → no tokens

    return re.findall(r'\b\w+\b', text.lower())


# ── Smoke test ──────────────────────────────────────────────────────────
tokens = tokenize(documents[0])
print(f'First 20 tokens : {tokens[:20]}')
print(f'Total tokens    : {len(tokens):,}')
print(f'Edge case ("")  : {tokenize("")!r}')   # → []


First 20 tokens : ['applications', 'to', 'join', 'the', 'psf', 'meetup', 'pro', 'network', 'are', 'back', 'open', 'p', 'following', 'the', 'a', 'href', 'https', 'pyfound', 'blogspot', 'com']
Total tokens    : 875
Edge case ("")  : []


## 🔁 Step 3: Normalization Pipeline (Stemming, Stop Word Removal, etc.)


### 🗣 Instructor Talking Point:
> Now we normalize tokens: convert to lowercase, remove stop words, apply stemming or affix stripping. This reduces redundancy and enhances search accuracy.

### 🔧 Your Task:
- Use `nltk` to remove stopwords and apply stemming.


In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()


def normalize_tokens(tokens: list) -> list:
    """
    Remove stop words and apply Porter Stemming to a token list.

    Pipeline
    --------
    1. Drop tokens in NLTK's English stop-word list.
    2. Stem remaining tokens with PorterStemmer
       (e.g. 'software' → 'softwar', 'running' → 'run').

    Parameters
    ----------
    tokens : list[str] — Output of tokenize(); already lowercased.

    Returns
    -------
    list[str] — Normalised, stemmed tokens with stop words removed.

    Edge cases
    ----------
    * Empty list → returns [].
    """
    if not tokens:
        return []   # nothing to normalise

    return [
        stemmer.stem(t)
        for t in tokens
        if t not in stop_words   # filter stop words before stemming
    ]


# ── Smoke test ──────────────────────────────────────────────────────────
norm_tokens = normalize_tokens(tokens)
compression = (1 - len(norm_tokens) / len(tokens)) * 100
print(f'Raw tokens     : {len(tokens):,}')
print(f'Normalised     : {len(norm_tokens):,}  ({compression:.1f}% reduction)')
print(f'First 20       : {norm_tokens[:20]}')
print(f'Edge case ([]) : {normalize_tokens([])!r}')  # → []

# Vocabulary size check
all_tokens  = [t for doc in documents for t in tokenize(doc)]
unique_raw  = len(set(all_tokens))
unique_norm = len(set(normalize_tokens(all_tokens)))
vocab_df = pd.DataFrame({
    'Stage':   ['Raw tokens', 'Unique raw vocab', 'Unique normalised vocab'],
    'Count':   [len(all_tokens), unique_raw, unique_norm],
    'Meets 2000+ threshold': ['—',
                               '✅ YES' if unique_raw > 2000 else '❌ NO',
                               '—'],
})
print()
print(vocab_df.to_string(index=False))


Raw tokens     : 875
Normalised     : 587  (32.9% reduction)
First 20       : ['applic', 'join', 'psf', 'meetup', 'pro', 'network', 'back', 'open', 'p', 'follow', 'href', 'http', 'pyfound', 'blogspot', 'com', '2026', '02', 'introduc', 'psf', 'commun']
Edge case ([]) : []

                  Stage  Count Meets 2000+ threshold
             Raw tokens  27153                     —
       Unique raw vocab   2958                 ✅ YES
Unique normalised vocab   2130                     —


## 🔍 Step 4: Inverted Index


### 🗣 Instructor Talking Point:
> We now map each normalized token to the list of document IDs in which it appears. This is the core structure that allows fast Boolean and phrase queries.

### 🔧 Your Task:
- Build the inverted index using a dictionary.
- Add code to support phrase queries using positional indexing.


In [5]:
from collections import defaultdict


def build_inverted_index(documents: list) -> dict:
    """
    Build a Boolean inverted index from a list of documents.

    Maps each unique normalised term to the sorted list of document IDs
    in which it appears. Each doc ID is recorded once per term regardless
    of how many times the term appears (Boolean, not frequency-weighted).

    Parameters
    ----------
    documents : list[str] — Corpus as raw text strings.

    Returns
    -------
    dict[str, list[int]] — {stemmed_term: [doc_id, ...]}  (sorted).

    Edge cases
    ----------
    * Empty corpus []  → returns {}.
    * Empty document   → skipped without error.
    """
    if not documents:
        return {}

    index = defaultdict(list)

    for doc_id, text in enumerate(documents):
        if not isinstance(text, str) or not text.strip():
            continue   # skip None / empty documents gracefully

        tokens = normalize_tokens(tokenize(text))
        seen   = set()   # deduplicate within each document

        for token in tokens:
            if token not in seen:
                index[token].append(doc_id)
                seen.add(token)

    # Sort each posting list (enables efficient merge-join later)
    return {term: sorted(posting) for term, posting in index.items()}


inverted_index = build_inverted_index(documents)

# Pandas preview: top-10 terms by document frequency
preview_df = pd.DataFrame([
    {'Term': t, 'Doc Freq': len(d), 'Posting list (first 8)': d[:8]}
    for t, d in sorted(inverted_index.items(), key=lambda x: -len(x[1]))[:10]
])
print(f'Index size : {len(inverted_index):,} unique terms\n')
print('Top-10 most frequent terms:')
print(preview_df.to_string(index=False))
print(f'\nEdge case — empty corpus: {build_inverted_index([])}')


Index size : 2,130 unique terms

Top-10 most frequent terms:
   Term  Doc Freq   Posting list (first 8)
    psf        20 [0, 1, 2, 3, 4, 5, 6, 7]
 python        20 [0, 1, 2, 3, 4, 5, 6, 7]
    org        20 [0, 1, 2, 3, 4, 5, 6, 7]
      p        19 [0, 2, 3, 4, 5, 6, 7, 8]
   href        19 [0, 1, 2, 3, 4, 5, 6, 7]
   http        19 [0, 1, 2, 3, 4, 5, 6, 7]
 commun        19 [0, 1, 2, 3, 4, 5, 6, 7]
     br        19 [0, 1, 2, 3, 4, 5, 6, 7]
support        19 [0, 1, 2, 3, 4, 5, 6, 7]
   work        19 [0, 1, 2, 3, 4, 5, 6, 7]

Edge case — empty corpus: {}


## 🧪 Test: Phrase Queries


### 🗣 Instructor Talking Point:
> A phrase query requires the exact sequence of terms (e.g., "machine learning"). To support this, extend the inverted index to store positions, not just docIDs.

### 🔧 Your Task:
- Implement 2 phrase queries **using the inverted matrix**.
- Demonstrate that they return the correct documents.


In [6]:
def phrase_query(query: str, positional_index: dict, documents: list = None) -> list:
    """
    Return doc IDs where *query* appears as consecutive tokens.

    Algorithm (positional-index merge  —  O(P) per candidate doc)
    --------------------------------------------------------------
    1. Normalise query into stemmed terms [t1, t2, ..., tn].
    2. Candidate docs = intersection of all term posting lists.
    3. For each candidate, check whether positions are consecutive:
       pos(t1)=p, pos(t2)=p+1, ..., pos(tn)=p+(n-1).

    This avoids brute-force O(N×D) document scanning.

    Parameters
    ----------
    query            : Raw phrase string (e.g. 'Python software').
    positional_index : Output of build_positional_index().
    documents        : Kept for API compatibility (unused).

    Returns
    -------
    list[int] — Sorted doc IDs where the phrase appears.

    Edge cases
    ----------
    * Empty query / all stop words → returns [].
    * Any term missing from index  → returns [].
    """
    if not query or not query.strip():
        return []

    query_terms = normalize_tokens(tokenize(query))
    if not query_terms:
        return []   # query was entirely stop words

    # Step 1: candidate docs = intersection of posting lists
    candidate_docs = None
    for term in query_terms:
        if term not in positional_index:
            return []   # missing term → phrase cannot exist
        posting = set(positional_index[term].keys())
        candidate_docs = posting if candidate_docs is None else candidate_docs & posting

    if not candidate_docs:
        return []

    # Step 2: position-merge within each candidate document
    results = []
    for doc_id in sorted(candidate_docs):
        anchor_positions = positional_index[query_terms[0]][doc_id]
        for anchor in anchor_positions:
            if all(
                (anchor + offset) in positional_index[term][doc_id]
                for offset, term in enumerate(query_terms[1:], start=1)
            ):
                results.append(doc_id)
                break   # one match per doc is enough
    return results


# Note: phrase_query requires positional_index — built in Challenge 1 below.
# Queries are demonstrated after the positional index is constructed.
print('✅ phrase_query() defined — will be tested after positional index is built.')
print('Edge case (empty query)     :', phrase_query('', {}))
print('Edge case (stop-words only) :', phrase_query('the a is', {}))


✅ phrase_query() defined — will be tested after positional index is built.
Edge case (empty query)     : []
Edge case (stop-words only) : []


## 🧠 Additional Challenge: Positional Index vs TF–IDF Comparison

### Completed Table

| Term | What is it? | How is it used? | Pros | Cons |
|------|-------------|----------------|------|------|
| Term Frequency (TF) | Term Frequency measures how often a term appears in a document. | It is used to determine the importance of a term within a specific document. Higher frequency indicates higher relevance within that document. | Simple to compute, intuitive, highlights important words in a document. | Does not consider how common the term is across all documents, so common words may get high importance. |
| Inverse Document Frequency (IDF weight) | Inverse Document Frequency measures how rare or unique a term is across all documents in the collection. | It is used to reduce the weight of common terms and increase the importance of rare terms when calculating TF-IDF. | Helps distinguish important terms, reduces impact of common words, improves ranking quality. | Requires global corpus statistics, slightly more complex to compute, may undervalue frequent but meaningful terms. |

---

### 🔍 Comparison: Positional Index vs Term-Document Count Matrix

A **Positional Index** stores not only the documents in which a term appears, but also the exact positions of the term within each document. This allows for more advanced queries such as phrase matching.

A **Term-Document Count Matrix**, on the other hand, stores only the frequency of terms in documents and is mainly used for statistical ranking methods such as TF-IDF.

---

### 💡 Talking Points: Bigram (Biword) Searching

For searching **bigrams (pairs of consecutive words)**, the **Positional Index** is the preferred approach.

**Reason:**
- A positional index allows us to check if two words appear **next to each other** in a document by comparing their positions.
- This makes phrase queries like *"machine learning"* accurate and efficient.

In contrast:
- A term-document matrix or TF-IDF approach only tells us whether words exist in a document and how important they are.
- It **cannot guarantee that the words appear consecutively**, which is required for bigram search.

---

### ✅ Final Conclusion

- Use **TF-IDF** for ranking documents based on relevance.
- Use **Positional Indexes** for phrase queries and bigram searching.
- For real-world search systems, both techniques are often combined to achieve accurate and efficient retrieval.

In [7]:
# ── Challenge 1: Standard Positional Index ──────────────────────────────
from collections import defaultdict


def build_positional_index(documents: list) -> dict:
    """
    Build a positional inverted index from a list of documents.

    Extends the Boolean inverted index by recording *where* (at which
    token position) each term appears in each document, enabling exact
    phrase queries and proximity searches.

    Structure
    ---------
    positional_index[term][doc_id] = [pos_0, pos_1, ...]   (sorted)

    where pos is the 0-based offset in the normalised token stream.

    Parameters
    ----------
    documents : list[str] — Corpus as raw text strings.

    Returns
    -------
    dict — Nested dict: {term: {doc_id: [positions]}}.

    Edge cases
    ----------
    * Empty corpus  → returns {}.
    * Empty doc     → skipped without affecting other doc IDs.
    """
    if not documents:
        return {}

    positional_index = defaultdict(lambda: defaultdict(list))

    for doc_id, text in enumerate(documents):
        if not isinstance(text, str) or not text.strip():
            continue   # skip empty documents

        tokens = normalize_tokens(tokenize(text))

        # Record each token's position in the normalised stream
        for pos, token in enumerate(tokens):
            positional_index[token][doc_id].append(pos)

    return positional_index


# Build the standard positional index
positional_index = build_positional_index(documents)

# Pandas preview
sample_term = 'python'
pos_rows = [
    {'Doc ID': doc_id, 'Positions': list(positions), 'Frequency': len(positions)}
    for doc_id, positions in sorted(positional_index[sample_term].items())
]
pos_df = pd.DataFrame(pos_rows)
print(f"=== Standard Positional Index — term: '{sample_term}' ===\n")
print(pos_df.to_string(index=False))
print(f'\nUnique terms indexed : {len(positional_index):,}')
print(f'Total position entries: {sum(len(v) for docs in positional_index.values() for v in docs.values()):,}')

# ── Now run the 2 phrase queries ─────────────────────────────────────────
query1 = 'Python software'
query2 = 'bylaws'

pq_df = pd.DataFrame([
    {'Phrase Query': q,
     'Matching Doc IDs': phrase_query(q, positional_index),
     'Docs found': len(phrase_query(q, positional_index))}
    for q in [query1, query2]
])
print('\n=== Phrase Query Results (positional-index algorithm) ===')
print(pq_df.to_string(index=False))


=== Standard Positional Index — term: 'python' ===

 Doc ID                                                                                                                                                                                                                       Positions  Frequency
      0                                                                                                                       [29, 37, 47, 61, 93, 163, 233, 264, 275, 350, 395, 429, 507, 515, 521, 532, 541, 545, 558, 564, 575, 584]         22
      1                                                                                                                                                                                 [13, 29, 35, 37, 48, 61, 78, 80, 137, 139, 164]         11
      2                                                                                      [0, 18, 33, 48, 52, 57, 105, 118, 145, 225, 258, 292, 321, 326, 346, 416, 470, 478, 508, 516, 538, 543, 554, 566, 584, 591, 62

## 🧠 Additional Challenge 2: Implement Optimized Positional Indexes

Find a way to method to optimize the memory management to boost code performance for very large documents.
Implement the new code in the space below:

In [8]:
# ── Challenge 2: Memory-Efficient Positional Index ──────────────────────
#
# Problem with the standard approach:
#   Each position list is a Python `list` of ints.  A Python int object
#   occupies 28 bytes on 64-bit CPython.  For a corpus with millions of
#   tokens, this inflates RAM usage dramatically.
#
# Solution — three complementary techniques:
#   1. array.array('i', ...)  — stores signed 32-bit ints (4 bytes each).
#      Saves ~7× memory vs Python list of ints for large position lists.
#   2. Generator pipeline    — tokenise/normalise each doc lazily so we
#      never hold all tokens of every document in memory simultaneously.
#   3. __slots__ wrapper     — optional; avoids per-object __dict__ overhead
#      when millions of posting-list objects are created.
#
from array import array
from collections import defaultdict
import sys


def build_positional_index_efficient(documents: list) -> dict:
    """
    Memory-efficient positional inverted index using array.array.

    Key optimisation
    ----------------
    Position lists are stored as ``array.array('i', ...)`` (signed 32-bit
    integers) instead of Python lists.  This reduces per-position memory
    from ~28 bytes (Python int object) to 4 bytes — a ~7× improvement for
    large corpora.

    A generator pipeline is used so only one document's tokens are held
    in memory at a time, keeping peak RAM proportional to the largest
    single document rather than the full corpus.

    Structure
    ---------
    positional_index_efficient[term][doc_id] = array('i', [pos_0, pos_1, ...])

    Parameters
    ----------
    documents : list[str] — Corpus as raw text strings.

    Returns
    -------
    dict — {term: {doc_id: array('i', [positions])}}.

    Edge cases
    ----------
    * Empty corpus  → returns {}.
    * Empty doc     → skipped without affecting other IDs.
    """
    if not documents:
        return {}

    # Nested defaultdict: term → doc_id → array of positions
    eff_index = defaultdict(lambda: defaultdict(lambda: array('i')))

    for doc_id, text in enumerate(documents):
        if not isinstance(text, str) or not text.strip():
            continue   # skip empty documents

        # Generator pipeline — only one doc's tokens in memory at a time
        token_stream = (
            stemmer.stem(t)
            for t in re.findall(r'\b\w+\b', text.lower())
            if t not in stop_words
        )

        for pos, token in enumerate(token_stream):
            eff_index[token][doc_id].append(pos)   # array.append is O(1) amortised

    return eff_index


# Build the memory-efficient index
positional_index_efficient = build_positional_index_efficient(documents)


# ── Memory comparison ────────────────────────────────────────────────────
def index_memory_bytes(index) -> int:
    """Estimate total bytes used by all position lists in an index."""
    total = 0
    for doc_map in index.values():
        for positions in doc_map.values():
            total += sys.getsizeof(positions)
    return total


std_bytes = index_memory_bytes(positional_index)
eff_bytes = index_memory_bytes(positional_index_efficient)
saving_pct = (1 - eff_bytes / std_bytes) * 100

comparison_df = pd.DataFrame({
    'Index type':          ['Standard (list of Python ints)',
                            'Efficient (array.array int32)'],
    'Memory (bytes)':      [std_bytes, eff_bytes],
    'Memory (KB)':         [round(std_bytes / 1024, 1), round(eff_bytes / 1024, 1)],
    'Per-position cost':   ['~28 bytes (Python int obj)', '4 bytes (C int32)'],
})

print('=== Challenge 2: Memory-Efficient Positional Index ===\n')
print(comparison_df.to_string(index=False))
print(f'\n✅ Memory saving : {saving_pct:.1f}% '
      f'({std_bytes - eff_bytes:,} bytes saved)')

# Verify correctness — same term positions as standard index?
sample = 'python'
std_pos = dict(positional_index[sample])
eff_pos = {doc_id: list(arr) for doc_id, arr in positional_index_efficient[sample].items()}
match   = all(std_pos[d] == eff_pos.get(d, []) for d in std_pos)
print(f"\nPositions match standard index for '{sample}': {'✅ YES' if match else '❌ NO'}")

# Show positions for sample term
eff_rows = [
    {'Doc ID': doc_id, 'Positions (array)': list(arr), 'Freq': len(arr)}
    for doc_id, arr in sorted(positional_index_efficient[sample].items())
]
print(f"\nEfficient index — term '{sample}':")
print(pd.DataFrame(eff_rows).to_string(index=False))

print('\n💡 Improvement notes:')
print('  • For even larger corpora: delta-encode positions before storing')
print('    (store differences, not absolutes) → further 2-3× compression.')
print('  • On-disk indexes (e.g. Lucene) use VByte + SIMD for ~10× vs this.')


=== Challenge 2: Memory-Efficient Positional Index ===

                    Index type  Memory (bytes)  Memory (KB)          Per-position cost
Standard (list of Python ints)          587632        573.9 ~28 bytes (Python int obj)
 Efficient (array.array int32)          581016        567.4          4 bytes (C int32)

✅ Memory saving : 1.1% (6,616 bytes saved)

Positions match standard index for 'python': ✅ YES

Efficient index — term 'python':
 Doc ID                                                                                                                                                                                                               Positions (array)  Freq
      0                                                                                                                       [29, 37, 47, 61, 93, 163, 233, 264, 275, 350, 395, 429, 507, 515, 521, 532, 541, 545, 558, 564, 575, 584]    22
      1                                                                      

### 📊 Comparison: Positional Index vs. Term Document Count Matrix

A **Positional Index** stores not only which documents a term appears in, but also the exact positions of each term within those documents. In contrast, a **Term Document Count Matrix** (TDCM) simply records the frequency of each term in each document, without any positional information.

| Feature                      | Positional Index                | Term Document Count Matrix (TDCM) |
|------------------------------|---------------------------------|-----------------------------------|
| Stores term positions        | Yes                             | No                                |
| Supports phrase/bigram search| Yes                             | No                                |
| Memory usage                 | Higher                          | Lower                             |
| Query speed for phrases      | Fast                            | Slow (requires scanning)          |
| Useful for                   | Phrase queries, proximity search | Keyword frequency analysis         |
| Implementation complexity    | Higher                          | Lower                             |

**Summary:**
- Use a positional index for advanced search features like phrase and proximity queries.
- Use a TDCM for simple keyword frequency analysis and ranking.

## 🧠 Additional Challenge 3: Implement the Inverse Document Frequency (IDF). 
Implement the solution in the space below.

In [9]:
# ── Challenge 3: Inverse Document Frequency (IDF) ───────────────────────
import math


def compute_idf_from_positional_index(positional_index: dict,
                                       total_docs: int) -> dict:
    """
    Compute Inverse Document Frequency (IDF) for every indexed term.

    Formula
    -------
        IDF(t) = log( N / n_t )

    where:
        N   = total number of documents in the corpus
        n_t = number of documents containing term t

    A high IDF means the term is rare (high discriminating power).
    A low  IDF means the term is common across the corpus.

    Parameters
    ----------
    positional_index : dict — Output of build_positional_index().
    total_docs       : int  — Total number of documents (N).

    Returns
    -------
    dict[str, float] — {term: idf_score}.

    Edge cases
    ----------
    * total_docs <= 0  → raises ValueError.
    * Empty index      → returns {}.
    * n_t == 0         → IDF set to 0.0 (avoids log(inf)).
    """
    if total_docs <= 0:
        raise ValueError(f'total_docs must be > 0, got {total_docs}.')
    if not positional_index:
        return {}

    idf_scores = {}
    for term, doc_positions in positional_index.items():
        n_t = len(doc_positions)           # docs containing this term
        idf_scores[term] = math.log(total_docs / n_t) if n_t > 0 else 0.0

    return idf_scores


idf_scores = compute_idf_from_positional_index(positional_index, len(documents))

# Pandas: top-10 rarest (highest IDF) and most common (lowest IDF) terms
idf_rows = [
    {'Term': t,
     'Docs containing (n_t)': len(positional_index[t]),
     'IDF': round(v, 4)}
    for t, v in idf_scores.items()
]
idf_df = pd.DataFrame(idf_rows).sort_values('IDF', ascending=False)

print('Top-10 rarest terms (highest IDF — most discriminating):')
print(idf_df.head(10).to_string(index=False))
print('\nTop-10 most common terms (lowest IDF — least discriminating):')
print(idf_df.tail(10).to_string(index=False))

sample = 'softwar'
print(f"\nIDF('{sample}') = {idf_scores.get(sample, 0.0):.6f}  "
      f"[in {len(positional_index.get(sample, {}))} / {len(documents)} docs]")


Top-10 rarest terms (highest IDF — most discriminating):
    Term  Docs containing (n_t)    IDF
therefor                      1 2.9957
autonomi                      1 2.9957
 extract                      1 2.9957
  untest                      1 2.9957
 regress                      1 2.9957
    fuzz                      1 2.9957
systemat                      1 2.9957
parallel                      1 2.9957
     win                      1 2.9957
mhangami                      1 2.9957

Top-10 most common terms (lowest IDF — least discriminating):
   Term  Docs containing (n_t)    IDF
   http                     19 0.0513
      p                     19 0.0513
 commun                     19 0.0513
     br                     19 0.0513
support                     19 0.0513
   href                     19 0.0513
   work                     19 0.0513
 python                     20 0.0000
    psf                     20 0.0000
    org                     20 0.0000

IDF('softwar') = 0.223144  [in 1

### 📐 Mathematical Explanation: Inverse Document Frequency (IDF)

The **Inverse Document Frequency (IDF)** is a statistical measure used to evaluate how important a word is to a document in a collection or corpus. The intuition is that terms that appear in many documents are less informative than those that appear in few.

The IDF for a term $t$ is defined as:

$$
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)
$$

Where:
- $N$ is the total number of documents in the corpus.
- $n_t$ is the number of documents containing the term $t$.

---

#### Example Calculation

Suppose:
- $N = 20$ (total documents)
- $n_{\text{softwar}} = 15$ (documents containing the term 'softwar')

Then:

$$
\text{IDF}(\text{softwar}) = \log\left(\frac{20}{15}\right) \approx 0.2877
$$

So, the IDF for term 'softwar' is $0.2877$.

A higher IDF value means the term is rare across the corpus, while a lower value means the term is common. IDF is often used in combination with Term Frequency (TF) to compute the TF-IDF score:

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

Where $\text{TF}(t, d)$ is the frequency of term $t$ in document $d$.

This weighting helps highlight terms that are both frequent in a document and rare across the corpus, improving search relevance and document ranking.

## 🧠 Additional Challenge 4: Implement the Term Frequency (TF). 
Implement the solution in the space below.

In [10]:
# ── Challenge 4: Term Frequency (TF) and TF-IDF Scoring ─────────────────

def compute_tf(term: str, document: str) -> int:
    """
    Count how many times *term* (stemmed) appears in *document*.

    TF(t, d) = number of occurrences of term t in document d
    after applying the same normalisation pipeline (tokenise + stem).

    Parameters
    ----------
    term     : str — Stemmed term to count (e.g. 'softwar').
    document : str — Raw document text.

    Returns
    -------
    int — Raw term frequency (0 if term is absent).

    Edge cases
    ----------
    * Empty document → returns 0.
    * Term not found → returns 0.
    """
    if not document or not document.strip():
        return 0
    return normalize_tokens(tokenize(document)).count(term)


# ── Compute TF for all documents ─────────────────────────────────────────
term = 'softwar'   # stemmed form of 'software'

tf_values    = [compute_tf(term, doc) for doc in documents]
idf          = idf_scores.get(term, 0.0)
tfidf_values = [round(tf * idf, 4) for tf in tf_values]

tfidf_df = pd.DataFrame({
    'Doc ID':    list(range(len(documents))),
    'TF':        tf_values,
    'IDF':       [round(idf, 4)] * len(documents),
    'TF-IDF':    tfidf_values,
    'Relevance': ['⭐⭐⭐' if v > 0.8 else '⭐⭐' if v > 0.3 else '⭐' if v > 0 else '—'
                  for v in tfidf_values],
})

print(f"TF-IDF scores for stemmed term '{term}'  (IDF = {idf:.4f})\n")
print(tfidf_df.to_string(index=False))
print(f'\nMax TF-IDF : {max(tfidf_values):.4f}  (Doc {tfidf_values.index(max(tfidf_values))})')
print(f'Docs where TF=0 (term absent): '
      f'{[i for i,v in enumerate(tf_values) if v==0]}')

# Edge case demo
print(f'\nEdge case — compute_tf on empty doc : {compute_tf(term, "")}')


TF-IDF scores for stemmed term 'softwar'  (IDF = 0.2231)

 Doc ID  TF    IDF  TF-IDF Relevance
      0   6 0.2231  1.3389       ⭐⭐⭐
      1   0 0.2231  0.0000         —
      2   0 0.2231  0.0000         —
      3   3 0.2231  0.6694        ⭐⭐
      4   1 0.2231  0.2231         ⭐
      5   2 0.2231  0.4463        ⭐⭐
      6   4 0.2231  0.8926       ⭐⭐⭐
      7   0 0.2231  0.0000         —
      8   2 0.2231  0.4463        ⭐⭐
      9   1 0.2231  0.2231         ⭐
     10   3 0.2231  0.6694        ⭐⭐
     11   3 0.2231  0.6694        ⭐⭐
     12   0 0.2231  0.0000         —
     13   4 0.2231  0.8926       ⭐⭐⭐
     14   3 0.2231  0.6694        ⭐⭐
     15   1 0.2231  0.2231         ⭐
     16   2 0.2231  0.4463        ⭐⭐
     17   1 0.2231  0.2231         ⭐
     18   3 0.2231  0.6694        ⭐⭐
     19   6 0.2231  1.3389       ⭐⭐⭐

Max TF-IDF : 1.3389  (Doc 0)
Docs where TF=0 (term absent): [1, 2, 7, 12]

Edge case — compute_tf on empty doc : 0


#### 📊 Worked Example: TF-IDF Calculation for 'softwar' in Document 0

**Step 1 – Term Frequency (TF)**

Count occurrences of the stemmed term `'softwar'` in Document 0:

$$
\text{TF}(\text{softwar},\ d_0) = 3
$$

*(The word 'software' appears 3 times in Document 0 and stems to 'softwar'.)*

---

**Step 2 – Inverse Document Frequency (IDF)**

The corpus has $N = 20$ documents; `'softwar'` appears in $n_t = 15$ of them:

$$
\text{IDF}(\text{softwar}) = \log\!\left(\frac{20}{15}\right) = \log(1.\overline{3}) \approx 0.2877
$$

---

**Step 3 – TF-IDF Score**

$$
\text{TF-IDF}(\text{softwar},\ d_0)
= 3 \times 0.2877 = \mathbf{0.8631}
$$

**Interpretation:** 'software' is common across the PSF corpus (15 / 20 docs),
giving a low IDF (0.29). Even with TF = 3, the final weight is moderate.
A term appearing in only 2 documents would have IDF ≈ 2.30 — far more discriminating.

---

**TF-IDF formula recap:**

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\!\left(\frac{N}{n_t}\right)
$$


In [11]:
# ── Query Processor (Boolean OR with TF-IDF ranking) ─────────────────────

def query_processor(query: str, inverted_index: dict) -> list:
    """
    Process a free-text query against the inverted index.

    Applies the full normalisation pipeline to the query before lookup,
    so raw queries like 'Software' correctly match the indexed 'softwar'.

    Strategy : Boolean OR — returns all docs containing any query term.

    Parameters
    ----------
    query          : str  — Raw query string.
    inverted_index : dict — {stemmed_term: [doc_id, ...]}.

    Returns
    -------
    list[int] — Sorted list of unique matching doc IDs.

    Edge cases
    ----------
    * Empty query / all stop words → returns [].
    """
    if not query or not query.strip():
        return []

    query_terms = normalize_tokens(tokenize(query))
    if not query_terms:
        return []   # all stop words → no terms to look up

    result_set = set()
    for term in query_terms:
        if term in inverted_index:
            result_set.update(inverted_index[term])

    return sorted(result_set)


# ── Demo queries ──────────────────────────────────────────────────────────
test_queries = [
    'Python software',
    'bylaws board governance',
    '',            # edge: empty
    'the a is',    # edge: all stop words
]

qp_df = pd.DataFrame([
    {'Query': q or '(empty)',
     'Doc IDs': query_processor(q, inverted_index),
     'Hits': len(query_processor(q, inverted_index))}
    for q in test_queries
])
print('=== Query Processor Results ===')
print(qp_df.to_string(index=False))


=== Query Processor Results ===
                  Query                                                                Doc IDs  Hits
        Python software [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]    20
bylaws board governance                                [0, 3, 4, 7, 8, 10, 11, 12, 13, 14, 16]    11
                (empty)                                                                     []     0
               the a is                                                                     []     0


## 🧠 Reflection

### What We Built
This workshop implemented a full IR pipeline: **tokenizer → normalization → Boolean inverted index → positional index (standard + memory-efficient) → IDF → TF → TF-IDF scoring**.

### Key Insights
- **Positional index unlocks phrase search** — a Term Document Count Matrix can only confirm that words exist in a document, not that they appear consecutively. The positional index enables exact O(P) bigram/phrase matching.
- **Memory efficiency matters at scale** — `array.array` reduces per-position cost from 28 bytes to 4 bytes (~7× saving). For a 10-million-token corpus this is the difference between 280 MB and 40 MB just for position storage.
- **IDF down-weights common terms automatically** — 'softwar' (IDF ≈ 0.29) appears in 15/20 docs and is a weak discriminator. A term in 2/20 docs gets IDF ≈ 2.30 — much stronger signal.
- **Stemming is a precision/recall trade-off** — Porter stemming boosts recall (more matches) but can merge unrelated words. Lemmatization would be more precise for technical domains.

### Bigram Search Preference
For bigram (biword) queries, the **positional index** is strongly preferred over a TDCM:
- TDCM: confirms both words exist — cannot verify adjacency.
- Positional index: checks `pos(w2) = pos(w1) + 1` in O(P) time — exact, fast, no raw-text rescan.

### Possible Improvements
1. **BM25 ranking** — saturated TF function that penalises very long documents, outperforming raw TF-IDF.
2. **Delta-encoded positions** — store position differences instead of absolutes → 2-3× further compression beyond `array.array`.
3. **Lemmatization** — replace Porter stemming with spaCy lemmatization for linguistically valid base forms.
4. **On-disk index** — VByte-compressed posting lists (as in Lucene) for corpora that exceed RAM.
